In [10]:
#Loading necessary libraries
from keras.models import Sequential
from keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import load_img
from keras.layers import Dense, Conv2D, Flatten, MaxPooling2D, Dropout
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from PIL import Image

In [11]:
train_dir = '/kaggle/input/emotiond/train'
test_dir = '/kaggle/input/emotiond/test'

In [12]:
# Function to create a DataFrame with image paths and labels
def createdf(dir):
    img_path = []
    labels = []
    for label in os.listdir(dir):
        for imgname in os.listdir(os.path.join(dir, label)):
            img_path.append(os.path.join(dir, label, imgname))
            labels.append(label)
        print(label, "completed")
    return img_path, labels

#
from tqdm.notebook import tqdm
def extractfeatures(image_path):
    features = []
    for image in tqdm(image_path):
        img = load_img(image, color_mode='grayscale')
        img = np.array(img)
        features.append(img)
    features = np.array(features)
    features = features.reshape(len(features),48,48,1)
    return features

In [13]:
#Creating DataFrame for training data
train = pd.DataFrame()
train['img_path'], train['label'] = createdf(train_dir)

#Creating DataFrame for test data
test = pd.DataFrame()
test['img_path'], test['label'] = createdf(test_dir)

fearful completed
disgusted completed
angry completed
neutral completed
sad completed
surprised completed
happy completed
fearful completed
disgusted completed
angry completed
neutral completed
sad completed
surprised completed
happy completed


In [14]:
train_features = extractfeatures(train['img_path'])
test_features = extractfeatures(test['img_path'])

  0%|          | 0/28709 [00:00<?, ?it/s]

  0%|          | 0/7178 [00:00<?, ?it/s]

In [15]:
x_train = train_features/255.0
x_test = test_features/255.0

le = LabelEncoder()
le.fit(train['label'])

LabelEncoder()

In [16]:
y_train = le.transform(train['label'])
y_test = le.transform(test['label'])

y_train = to_categorical(y_train, num_classes=7)
y_test = to_categorical(y_test, num_classes=7)

In [18]:
#Making the model
model = Sequential()

#convolutional layers
model.add(Conv2D(128, kernel_size=(3,3), activation='relu', input_shape=(48,48,1)))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.4))

model.add(Conv2D(256, kernel_size=(3,3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.4))

model.add(Conv2D(512, kernel_size=(3,3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.4))

model.add(Flatten())

#fully connected layers
model.add(Dense(512, activation='relu'))
model.add(Dropout(0.4))
model.add(Dense(256, activation='relu'))
model.add(Dropout(0.3))

#output layer
model.add(Dense(7, activation='softmax'))

#compiling Model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [21]:
model.fit(x= x_train,y = y_train, epochs=100, batch_size=128, validation_data=(x_test, y_test))

Epoch 1/100
225/225 ━━━━━━━━━━━━━━━━━━━━ 11s 49ms/step - accuracy: 0.7410 - loss: 0.7127 - val_accuracy: 0.6225 - val_loss: 1.0752
Epoch 2/100
225/225 ━━━━━━━━━━━━━━━━━━━━ 11s 49ms/step - accuracy: 0.7476 - loss: 0.7001 - val_accuracy: 0.6209 - val_loss: 1.0672
Epoch 3/100
225/225 ━━━━━━━━━━━━━━━━━━━━ 11s 49ms/step - accuracy: 0.7397 - loss: 0.6945 - val_accuracy: 0.6243 - val_loss: 1.0903
Epoch 4/100
225/225 ━━━━━━━━━━━━━━━━━━━━ 11s 49ms/step - accuracy: 0.7504 - loss: 0.6750 - val_accuracy: 0.6208 - val_loss: 1.0801
Epoch 5/100
225/225 ━━━━━━━━━━━━━━━━━━━━ 11s 49ms/step - accuracy: 0.7580 - loss: 0.6592 - val_accuracy: 0.6187 - val_loss: 1.0795
Epoch 6/100
225/225 ━━━━━━━━━━━━━━━━━━━━ 11s 48ms/step - accuracy: 0.7543 - loss: 0.6695 - val_accuracy: 0.6234 - val_loss: 1.0812
Epoch 7/100
225/225 ━━━━━━━━━━━━━━━━━━━━ 11s 48ms/step - accuracy: 0.7633 - loss: 0.6600 - val_accuracy: 0.6236 - val_loss: 1.0926
Epoch 8/100
225/225 ━━━━━━━━━━━━━━━━━━━━ 11s 48ms/step - accuracy: 0.7604 - loss: 0

In [23]:
model_json = model.to_json()
with open("emotion_detector.json", "w") as json_file:
    json_file.write(model_json)
model.save("emotion_detector.h5")